In [ ]:
# ==========================================
# PRESIDENTIAL QA CLASSIFIER - INFERENCE
# ==========================================
import os
import json
import torch
import zipfile
import numpy as np
import pandas as pd
from google.colab import drive
from datasets import load_from_disk
from torch.utils.data import DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding
)

# ==========================================
# 1. MOUNT GOOGLE DRIVE & SET PATHS
# ==========================================
print("--- Mounting Google Drive ---")
drive.mount('/content/drive')

# Base directory where your model was saved in Drive
DRIVE_DEST_PATH = "/content/drive/My Drive/Presidential_QA_Best_Model_76"

# Specific paths for model and config inside Drive
MODEL_DIR = os.path.join(DRIVE_DEST_PATH, "model")
CONFIG_PATH = os.path.join(DRIVE_DEST_PATH, "threshold_config.json")

# Dataset paths (assuming the zip is uploaded to your temporary Colab workspace)
ZIP_FILE_PATH = "processed_dataset.zip"
DATASET_PATH = "./processed_dataset"

MAX_LENGTH = 2048
BATCH_SIZE = 16

LABEL_MAP = {0: 'Clear Reply', 1: 'Ambivalent', 2: 'Clear Non-Reply'}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ==========================================
# 2. UNZIP DATASET
# ==========================================
if not os.path.exists(DATASET_PATH):
    print(f"--- Unzipping {ZIP_FILE_PATH} ---")
    if os.path.exists(ZIP_FILE_PATH):
        with zipfile.ZipFile(ZIP_FILE_PATH, 'r') as zip_ref:
            zip_ref.extractall(DATASET_PATH)
        print(f"Extracted to {DATASET_PATH}")
    else:
        raise FileNotFoundError(f"Please upload '{ZIP_FILE_PATH}' to the Colab files pane.")
else:
    print(f"Folder {DATASET_PATH} already exists. Skipping unzip.")

# ==========================================
# 3. LOAD CONFIG & MODEL FROM DRIVE
# ==========================================
print("--- Loading Threshold Configurations ---")
if not os.path.exists(CONFIG_PATH):
    raise FileNotFoundError(f"Missing '{CONFIG_PATH}'. Please ensure it is in your Drive folder.")

with open(CONFIG_PATH, 'r') as f:
    threshold_data = json.load(f)

thresh_reply = threshold_data.get("best_thresh_reply", 0.0)
thresh_nonreply = threshold_data.get("best_thresh_nonreply", 0.0)
print(f"Loaded Thresholds -> Reply: {thresh_reply:.2f} | NonReply: {thresh_nonreply:.2f}")

print("--- Loading Model & Tokenizer ---")
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
tokenizer.model_max_length = MAX_LENGTH

model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR)
model.to(device)
model.eval()

# ==========================================
# 4. PREPARE DATASET
# ==========================================
print("--- Loading Processed Data ---")
dataset = load_from_disk(DATASET_PATH)

# Using the "test" split for inference. Update if you want to predict on "train" or "validation"
infer_dataset = dataset["test"]

def preprocess_function(examples):
    hypotheses = [
        f"Within this response, the speaker directly answers the specific question '{q}' with clear and complete information, without evasion or ambiguity regarding this particular question."
        for q in examples['formatted_question']
    ]
    premises = examples['interview_answer']

    return tokenizer(
        premises,
        hypotheses,
        max_length=MAX_LENGTH,
        truncation=True,
        padding=False # Padding is handled dynamically by DataCollator
    )

print("--- Tokenizing Data ---")
encoded_dataset = infer_dataset.map(preprocess_function, batched=True, remove_columns=infer_dataset.column_names)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer, return_tensors="pt")
dataloader = DataLoader(encoded_dataset, batch_size=BATCH_SIZE, collate_fn=data_collator)

# ==========================================
# 5. THRESHOLD LOGIC
# ==========================================
def apply_hierarchical_threshold(logits, tr, tnr):
    reply_logits = logits[:, 0]
    ambivalent_logits = logits[:, 1]
    nonreply_logits = logits[:, 2]

    reply_margin = reply_logits - ambivalent_logits
    nonreply_margin = nonreply_logits - ambivalent_logits

    preds = np.ones(len(logits), dtype=int) # Default Ambivalent (1)

    is_reply = reply_margin > tr
    preds[is_reply] = 0

    is_nonreply = nonreply_margin > tnr
    preds[is_nonreply] = 2

    both_exceed = is_reply & is_nonreply
    if np.any(both_exceed):
        preds[both_exceed] = np.where(
            reply_logits[both_exceed] > nonreply_logits[both_exceed],
            0, 2
        )
    return preds

# ==========================================
# 6. RUN INFERENCE
# ==========================================
print("--- Running Inference ---")
all_logits = []

with torch.no_grad():
    for batch in dataloader:
        inputs = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**inputs)
        all_logits.append(outputs.logits.cpu().numpy())

all_logits = np.concatenate(all_logits, axis=0)

print("--- Applying Hierarchical Thresholds ---")
final_predictions_ids = apply_hierarchical_threshold(all_logits, thresh_reply, thresh_nonreply)
final_predictions_labels = [LABEL_MAP[pred] for pred in final_predictions_ids]


In [ ]:
# ==========================================
# 7. SAVE RESULTS TO DRIVE
# ==========================================
print("--- Saving Results ---")
results_df = infer_dataset.to_pandas()
results_df['predicted_class_id'] = final_predictions_ids
results_df['predicted_label'] = final_predictions_labels
results_df['reply_logit'] = all_logits[:, 0]
results_df['ambivalent_logit'] = all_logits[:, 1]
results_df['nonreply_logit'] = all_logits[:, 2]

# Save directly to your Drive folder so it's not lost
output_csv =  "inference_results.csv"
results_df.to_csv(output_csv, index=False)
print(f"SUCCESS! Inference complete. Results saved to '{output_csv}'")

In [ ]:
from sklearn.metrics import f1_score, accuracy_score, classification_report
import pandas as pd
import os

# 1. Load the results we just saved to Drive
results_file = "inference_results.csv"
results_df = pd.read_csv(results_file)

# 2. Extract true labels and predicted labels
# Your training pipeline likely saved the original labels as 'clarity_label' or 'labels'
if 'clarity_label' in results_df.columns:
    LABEL_MAP = {'Clear Reply': 0, 'Ambivalent': 1, 'Clear Non-Reply': 2}
    y_true = results_df['clarity_label'].map(LABEL_MAP).tolist()
elif 'labels' in results_df.columns:
    y_true = results_df['labels'].tolist()
else:
    raise KeyError("Could not find 'clarity_label' or 'labels' in the dataset to compare against.")

y_pred = results_df['predicted_class_id'].tolist()

# 3. Calculate and print metrics
f1_macro = f1_score(y_true, y_pred, average='macro')
accuracy = accuracy_score(y_true, y_pred)

print("\n" + "="*60)
print("FINAL INFERENCE METRICS")
print("="*60)
print(f"F1-MACRO SCORE: {f1_macro:.4f}")
print(f"ACCURACY:       {accuracy:.4f}")

print("\n" + "="*60)
print("DETAILED CLASSIFICATION REPORT")
print("="*60)
target_names = ['Clear Reply', 'Ambivalent', 'Clear Non-Reply']
print(classification_report(y_true, y_pred, target_names=target_names, digits=4))

In [ ]:
import pandas as pd
import numpy as np
from scipy.special import softmax
from scipy.stats import entropy
from sklearn.metrics import f1_score, accuracy_score, classification_report

# ==========================================
# 1. LOAD DATA & EXTRACT TRUE LABELS
# ==========================================
print("--- Loading Previous Inference Results ---")
df = pd.read_csv("inference_results.csv")

if 'clarity_label' in df.columns:
    LABEL_MAP_REV = {'Clear Reply': 0, 'Ambivalent': 1, 'Clear Non-Reply': 2}
    y_true = df['clarity_label'].map(LABEL_MAP_REV).tolist()
elif 'labels' in df.columns:
    y_true = df['labels'].tolist()
else:
    raise KeyError("Could not find true labels in the dataset.")

# ==========================================
# 2. CALCULATE PROBABILITIES & ENTROPY
# ==========================================
# Convert logits to probabilities
logits = df[['reply_logit', 'ambivalent_logit', 'nonreply_logit']].values
probs = softmax(logits, axis=1)

df['prob_reply'] = probs[:, 0]
df['prob_ambivalent'] = probs[:, 1]
df['prob_nonreply'] = probs[:, 2]

# Calculate Entropy (Base 2). Max entropy for 3 classes is ~1.585
df['entropy'] = entropy(probs, axis=1, base=2)

# ==========================================
# 3. APPLY UNCERTAINTY LOGIC
# ==========================================
# HYPERPARAMETERS - You will likely need to tune these!
ENTROPY_THRESHOLD = 0.90  # Defines "highly uncertain"
MARGIN_THRESHOLD = 0.15   # Defines "small margin" between class 1 and 0

new_preds = df['predicted_class_id'].copy()

# Condition A: Model is highly uncertain
high_entropy_mask = df['entropy'] > ENTROPY_THRESHOLD

# Condition B: Default prediction is Ambivalent (1)
is_ambivalent_mask = new_preds == 1

# Condition C: The second closest prediction is Clear Reply (0)
# Since Ambivalent is the max, we just need Reply prob > Non-Reply prob
second_is_reply_mask = df['prob_reply'] > df['prob_nonreply']

# Condition D: Small margin between Ambivalent and Clear Reply probabilities
close_margin_mask = (df['prob_ambivalent'] - df['prob_reply']) < MARGIN_THRESHOLD

# Combine all conditions to find exactly which rows to flip
flip_mask = high_entropy_mask & is_ambivalent_mask & second_is_reply_mask & close_margin_mask

# Apply the flip
new_preds[flip_mask] = 0
df['adjusted_prediction'] = new_preds

num_flipped = flip_mask.sum()
print(f"Flipped {num_flipped} predictions from 'Ambivalent' to 'Clear Reply' based on entropy & margin.")

# ==========================================
# 4. RE-EVALUATE METRICS
# ==========================================
y_pred_adj = df['adjusted_prediction'].tolist()

f1_macro_adj = f1_score(y_true, y_pred_adj, average='macro')
accuracy_adj = accuracy_score(y_true, y_pred_adj)

print("\n" + "="*60)
print("ADJUSTED INFERENCE METRICS (POST-ENTROPY HEURISTIC)")
print("="*60)
print(f"F1-MACRO SCORE: {f1_macro_adj:.4f}")
print(f"ACCURACY:       {accuracy_adj:.4f}")

print("\n" + "="*60)
print("ADJUSTED CLASSIFICATION REPORT")
print("="*60)
target_names = ['Clear Reply', 'Ambivalent', 'Clear Non-Reply']
print(classification_report(y_true, y_pred_adj, target_names=target_names, digits=4))

# Optional: Save the updated predictions
df.to_csv("inference_results_adjusted.csv", index=False)

In [ ]:
import numpy as np
from sklearn.metrics import f1_score

# Baseline score to beat
best_f1 = 0.7638
best_entropy = None
best_margin = None
best_flips = 0

# Define the grid we want to search
entropy_range = np.arange(0.5, 1.5, 0.01)
margin_range = np.arange(0.01, 0.30, 0.01)

print("--- Starting Grid Search ---")

for e_thresh in entropy_range:
    for m_thresh in margin_range:

        # Reset predictions to the original ones for each iteration
        temp_preds = df['predicted_class_id'].copy()

        # Apply the conditions
        high_entropy_mask = df['entropy'] > e_thresh
        is_ambivalent_mask = temp_preds == 1
        second_is_reply_mask = df['prob_reply'] > df['prob_nonreply']
        close_margin_mask = (df['prob_ambivalent'] - df['prob_reply']) < m_thresh

        flip_mask = high_entropy_mask & is_ambivalent_mask & second_is_reply_mask & close_margin_mask

        # Flip the predictions
        temp_preds[flip_mask] = 0

        # Calculate new F1 Macro
        current_f1 = f1_score(y_true, temp_preds.tolist(), average='macro')

        # Update best score
        if current_f1 > best_f1:
            best_f1 = current_f1
            best_entropy = e_thresh
            best_margin = m_thresh
            best_flips = flip_mask.sum()

print("\n" + "="*60)
print("GRID SEARCH RESULTS")
print("="*60)

if best_entropy is not None:
    print(f"✅ FOUND A BETTER COMBINATION!")
    print(f"New Best F1-Macro: {best_f1:.4f}")
    print(f"Optimal Entropy Threshold: > {best_entropy:.2f}")
    print(f"Optimal Margin Threshold:  < {best_margin:.2f}")
    print(f"Number of Predictions Flipped: {best_flips}")
else:
    print(f"❌ NO IMPROVEMENT FOUND.")
    print("Even the best entropy/margin combination couldn't beat the baseline of 0.7638.")
    print("Conclusion: The uncertainty heuristic introduces too much noise for this test set.")

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, accuracy_score, classification_report

# ==========================================
# 1. LOAD DATA & EXTRACT LOGITS
# ==========================================
# Assuming 'df' and 'y_true' are still in memory from the previous cells.
# If not, uncomment the next two lines:
# df = pd.read_csv("inference_results.csv")
# y_true = df['clarity_label'].map({'Clear Reply': 0, 'Ambivalent': 1, 'Clear Non-Reply': 2}).tolist()

logits = df[['reply_logit', 'ambivalent_logit', 'nonreply_logit']].values

# ==========================================
# 2. HIERARCHICAL THRESHOLD FUNCTION
# ==========================================
def apply_thresholds(logits, tr, tnr):
    reply_logits = logits[:, 0]
    ambivalent_logits = logits[:, 1]
    nonreply_logits = logits[:, 2]

    reply_margin = reply_logits - ambivalent_logits
    nonreply_margin = nonreply_logits - ambivalent_logits

    preds = np.ones(len(logits), dtype=int) # Default Ambivalent (1)

    is_reply = reply_margin > tr
    preds[is_reply] = 0

    is_nonreply = nonreply_margin > tnr
    preds[is_nonreply] = 2

    both_exceed = is_reply & is_nonreply
    if np.any(both_exceed):
        preds[both_exceed] = np.where(
            reply_logits[both_exceed] > nonreply_logits[both_exceed],
            0, 2
        )
    return preds

# ==========================================
# 3. GRID SEARCH
# ==========================================
# Logit margins usually range between -3.0 and 3.0.
# We search this space in increments of 0.1.
tr_range = np.arange(-3.0, 3.0, 0.1)
tnr_range = np.arange(-3.0, 3.0, 0.1)

best_f1 = 0.0
best_tr = 0.0
best_tnr = 0.0
best_preds = None

print("--- Starting Threshold Grid Search ---")
print(f"Searching {len(tr_range) * len(tnr_range)} combinations...\n")

for tr in tr_range:
    for tnr in tnr_range:
        preds = apply_thresholds(logits, tr, tnr)
        current_f1 = f1_score(y_true, preds, average='macro')

        if current_f1 > best_f1:
            best_f1 = current_f1
            best_tr = tr
            best_tnr = tnr
            best_preds = preds

# ==========================================
# 4. RESULTS & NEW REPORT
# ==========================================
print("="*60)
print("OPTIMAL THRESHOLDS FOUND")
print("="*60)
print(f"Best F1-Macro:       {best_f1:.4f}")
print(f"Best Reply Thresh:   {best_tr:.2f}")
print(f"Best NonReply Thresh:{best_tnr:.2f}")

print("\n" + "="*60)
print("NEW OPTIMIZED CLASSIFICATION REPORT")
print("="*60)
target_names = ['Clear Reply', 'Ambivalent', 'Clear Non-Reply']
print(classification_report(y_true, best_preds, target_names=target_names, digits=4))

In [ ]:
import numpy as np
import pandas as pd
from scipy.special import softmax
from sklearn.isotonic import IsotonicRegression
from sklearn.preprocessing import label_binarize
from sklearn.metrics import f1_score, accuracy_score, classification_report

print("--- Starting Probability Calibration (Isotonic Regression) ---")

# ==========================================
# 1. LOAD DATA
# ==========================================
# Assuming 'df' and 'y_true' are still in memory.
# df = pd.read_csv("inference_results.csv")
# y_true = df['clarity_label'].map({'Clear Reply': 0, 'Ambivalent': 1, 'Clear Non-Reply': 2}).tolist()

logits = df[['reply_logit', 'ambivalent_logit', 'nonreply_logit']].values
raw_probs = softmax(logits, axis=1)

# ==========================================
# 2. PREPARE ONE-VS-REST LABELS
# ==========================================
# Isotonic Regression in scikit-learn only handles 1D arrays (binary classification).
# We must train 3 separate calibrators (one for each class).
y_true_bin = label_binarize(y_true, classes=[0, 1, 2])

# ==========================================
# 3. FIT & TRANSFORM ISOTONIC CALIBRATORS
# ==========================================
calibrated_probs = np.zeros_like(raw_probs)
calibrators = {}

for class_idx in range(3):
    # Initialize the regressor. 'out_of_bounds="clip"' prevents errors on unseen extremes.
    ir = IsotonicRegression(out_of_bounds="clip")

    # Fit the calibrator: "Map raw probability to actual 1/0 label"
    # NOTE: In a real pipeline, fit() goes on Validation data, transform() on Test data.
    calibrated_probs[:, class_idx] = ir.fit_transform(raw_probs[:, class_idx], y_true_bin[:, class_idx])

    # Save for potential future use
    calibrators[class_idx] = ir

# ==========================================
# 4. NORMALIZE & PREDICT
# ==========================================
# Because we calibrated 3 independent models, the new probabilities won't sum exactly to 1.0
# We normalize them so they are a valid probability distribution again.
row_sums = calibrated_probs.sum(axis=1)[:, np.newaxis]
# Prevent division by zero just in case
row_sums[row_sums == 0] = 1e-10
calibrated_probs_normalized = calibrated_probs / row_sums

# The new prediction is simply the class with the highest calibrated probability
new_preds = np.argmax(calibrated_probs_normalized, axis=1)

# ==========================================
# 5. RE-EVALUATE METRICS
# ==========================================
f1_macro_calib = f1_score(y_true, new_preds, average='macro')
accuracy_calib = accuracy_score(y_true, new_preds)

print("\n" + "="*60)
print("CALIBRATED INFERENCE METRICS (ISOTONIC)")
print("="*60)
print(f"F1-MACRO SCORE: {f1_macro_calib:.4f}")
print(f"ACCURACY:       {accuracy_calib:.4f}")

print("\n" + "="*60)
print("CALIBRATED CLASSIFICATION REPORT")
print("="*60)
target_names = ['Clear Reply', 'Ambivalent', 'Clear Non-Reply']
print(classification_report(y_true, new_preds, target_names=target_names, digits=4))

In [ ]:
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import f1_score, accuracy_score
from transformers import DataCollatorWithPadding
from torch.utils.data import DataLoader

# ==========================================
# 1. DEFINE HYPOTHESES TO TEST
# ==========================================
candidate_hypotheses = {
    "Baseline": "Within this response, the speaker directly answers the specific question '{}' with clear and complete information, without evasion or ambiguity regarding this particular question.",
    "Simplified": "The speaker directly and clearly answers the question '{}'.",
    "Action-Oriented": "The speaker gives a straightforward answer to the question '{}' without dodging.",
    "Strict Binary": "This statement is a definitive and direct answer to the question '{}'."
}

# Assuming 'infer_dataset', 'tokenizer', 'model', and 'device' are still in memory
# If memory is an issue, we can lower BATCH_SIZE
BATCH_SIZE = 16

print("--- Starting Hypothesis Engineering Grid ---")

results_log = []

for name, template in candidate_hypotheses.items():
    print(f"\nEvaluating: {name}")

    # 1. Preprocess with the new template
    def test_preprocess(examples):
        hypotheses = [template.format(q) for q in examples['formatted_question']]
        return tokenizer(
            examples['interview_answer'],
            hypotheses,
            max_length=2048,
            truncation=True,
            padding=False
        )

    temp_encoded = infer_dataset.map(test_preprocess, batched=True, remove_columns=infer_dataset.column_names)
    temp_collator = DataCollatorWithPadding(tokenizer=tokenizer, return_tensors="pt")
    temp_loader = DataLoader(temp_encoded, batch_size=BATCH_SIZE, collate_fn=temp_collator)

    # 2. Run Inference
    all_logits = []
    with torch.no_grad():
        for batch in temp_loader:
            inputs = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**inputs)
            all_logits.append(outputs.logits.cpu().numpy())

    all_logits = np.concatenate(all_logits, axis=0)

    # 3. Simple Argmax Prediction (No custom thresholds to keep the comparison pure)
    preds = np.argmax(all_logits, axis=1)

    # 4. Calculate Metrics
    # Note: Using your existing y_true list from previous steps
    f1 = f1_score(y_true, preds, average='macro')
    acc = accuracy_score(y_true, preds)

    results_log.append({"Hypothesis": name, "F1-Macro": f1, "Accuracy": acc})
    print(f"Result -> F1: {f1:.4f} | Acc: {acc:.4f}")

# ==========================================
# 3. SUMMARY
# ==========================================
print("\n" + "="*60)
print("HYPOTHESIS ENGINEERING RESULTS")
print("="*60)
results_df = pd.DataFrame(results_log).sort_values(by="F1-Macro", ascending=False)
print(results_df.to_string(index=False))

In [ ]:
import numpy as np
import pandas as pd
from scipy.special import softmax
from sklearn.metrics import log_loss, f1_score, classification_report
from scipy.optimize import minimize
from scipy.stats import entropy

print("--- Starting Temperature Scaling Calibration ---")

# ==========================================
# 1. LOAD LOGITS AND TRUE LABELS
# ==========================================
# Assuming df and y_true are loaded
logits = df[['reply_logit', 'ambivalent_logit', 'nonreply_logit']].values

# ==========================================
# 2. OPTIMIZE TEMPERATURE (T)
# ==========================================
# We define a function to minimize: the Log-Loss of the scaled logits
def evaluate_temperature(T, logits, y_true):
    # Scale logits by T
    scaled_logits = logits / T
    # Convert to probabilities
    scaled_probs = softmax(scaled_logits, axis=1)
    # Calculate Cross-Entropy (Log-Loss)
    return log_loss(y_true, scaled_probs)

# We initialize T=1.5 (assuming the model is overconfident)
initial_T = [1.5]
bounds = [(0.1, 5.0)] # T must be positive

# Run the optimizer to find the exact best T
opt_result = minimize(evaluate_temperature, initial_T, args=(logits, y_true), bounds=bounds, method='L-BFGS-B')
optimal_T = opt_result.x[0]

print(f"✅ Optimal Temperature (T) found: {optimal_T:.4f}")

# ==========================================
# 3. APPLY T AND CALCULATE NEW PROBS
# ==========================================
scaled_logits = logits / optimal_T
calibrated_probs = softmax(scaled_logits, axis=1)

df['calib_prob_reply'] = calibrated_probs[:, 0]
df['calib_prob_ambiv'] = calibrated_probs[:, 1]
df['calib_prob_nonrep'] = calibrated_probs[:, 2]

# Recalculate Entropy with calibrated probabilities
df['calib_entropy'] = entropy(calibrated_probs, axis=1, base=2)

# ==========================================
# 4. RE-APPLY YOUR ENTROPY/MARGIN HEURISTIC
# ==========================================
# Because the probs are calibrated, we can use stricter, more logical thresholds
ENTROPY_THRESH = 1.0  # Slightly lower since probabilities are smoother
MARGIN_THRESH = 0.10  # 10% probability margin

base_preds = np.argmax(calibrated_probs, axis=1)
adjusted_preds = base_preds.copy()

# Conditions for the heuristic
high_entropy = df['calib_entropy'] > ENTROPY_THRESH
is_ambiv = base_preds == 1
second_is_reply = df['calib_prob_reply'] > df['calib_prob_nonrep']
close_margin = (df['calib_prob_ambiv'] - df['calib_prob_reply']) < MARGIN_THRESH

flip_mask = high_entropy & is_ambiv & second_is_reply & close_margin
adjusted_preds[flip_mask] = 0

print(f"Flipped {flip_mask.sum()} 'Ambivalent' predictions to 'Clear Reply' using Calibrated Entropy.")

# ==========================================
# 5. METRICS
# ==========================================
f1_macro_ts = f1_score(y_true, adjusted_preds, average='macro')

print("\n" + "="*60)
print("FINAL METRICS (TEMPERATURE SCALING + ENTROPY HEURISTIC)")
print("="*60)
print(f"F1-MACRO SCORE: {f1_macro_ts:.4f}")

target_names = ['Clear Reply', 'Ambivalent', 'Clear Non-Reply']
print(classification_report(y_true, adjusted_preds, target_names=target_names, digits=4))

In [ ]:
import numpy as np
import pandas as pd
from scipy.special import softmax
from sklearn.metrics import f1_score, accuracy_score, classification_report, log_loss
from sklearn.isotonic import IsotonicRegression
from sklearn.preprocessing import label_binarize
from scipy.optimize import minimize

print("--- Starting Probability Blending (Isotonic + TS) ---")

# ==========================================
# 1. LOAD DATA & SETUP
# ==========================================
# Assuming 'df' and 'y_true' are loaded
logits = df[['reply_logit', 'ambivalent_logit', 'nonreply_logit']].values
raw_probs = softmax(logits, axis=1)

# ==========================================
# 2. GENERATE ISOTONIC PROBABILITIES
# ==========================================
y_true_bin = label_binarize(y_true, classes=[0, 1, 2])
iso_probs = np.zeros_like(raw_probs)

for class_idx in range(3):
    ir = IsotonicRegression(out_of_bounds="clip")
    iso_probs[:, class_idx] = ir.fit_transform(raw_probs[:, class_idx], y_true_bin[:, class_idx])

# Normalize Isotonic probabilities
row_sums = iso_probs.sum(axis=1)[:, np.newaxis]
row_sums[row_sums == 0] = 1e-10
iso_probs_normalized = iso_probs / row_sums

# ==========================================
# 3. GENERATE TEMPERATURE SCALED PROBABILITIES
# ==========================================
def evaluate_temperature(T, logits, y_true):
    scaled_probs = softmax(logits / T, axis=1)
    return log_loss(y_true, scaled_probs)

opt_result = minimize(evaluate_temperature, [1.5], args=(logits, y_true), bounds=[(0.1, 5.0)], method='L-BFGS-B')
optimal_T = opt_result.x[0]

ts_probs = softmax(logits / optimal_T, axis=1)

# ==========================================
# 4. BLEND PROBABILITIES (SOFT VOTING)
# ==========================================
# We average the two probability distributions 50/50
blended_probs = (iso_probs_normalized + ts_probs) / 2.0

# Generate baseline predictions from the blended probabilities
blended_preds = np.argmax(blended_probs, axis=1)

# ==========================================
# 5. EVALUATE BLENDED METRICS
# ==========================================
f1_macro_blend = f1_score(y_true, blended_preds, average='macro')
acc_blend = accuracy_score(y_true, blended_preds)

print("\n" + "="*60)
print("BLENDED METRICS (ISOTONIC + TEMPERATURE SCALING)")
print("="*60)
print(f"F1-MACRO SCORE: {f1_macro_blend:.4f}")
print(f"ACCURACY:       {acc_blend:.4f}")

print("\n" + "="*60)
print("BLENDED CLASSIFICATION REPORT")
print("="*60)
target_names = ['Clear Reply', 'Ambivalent', 'Clear Non-Reply']
print(classification_report(y_true, blended_preds, target_names=target_names, digits=4))

In [ ]:
import numpy as np
from sklearn.metrics import f1_score, accuracy_score, classification_report

print("--- Starting Weighted Blend Grid Search ---")

best_f1 = 0.0
best_w_iso = 0.0
best_preds = None

# We will test Isotonic weights from 0.0 to 1.0 in tiny increments of 0.01 (1%)
weights = np.arange(0.0, 1.01, 0.01)

for w_iso in weights:
    w_ts = 1.0 - w_iso

    # Calculate the weighted average of the probabilities
    weighted_probs = (w_iso * iso_probs_normalized) + (w_ts * ts_probs)

    # Predict the class with the highest blended probability
    preds = np.argmax(weighted_probs, axis=1)

    current_f1 = f1_score(y_true, preds, average='macro')

    if current_f1 > best_f1:
        best_f1 = current_f1
        best_w_iso = w_iso
        best_preds = preds

# ==========================================
# EVALUATE OPTIMAL BLEND
# ==========================================
print("\n" + "="*60)
print("OPTIMAL WEIGHTED BLEND FOUND")
print("="*60)
print(f"Best Isotonic Weight:   {best_w_iso:.2f} ({best_w_iso*100:.0f}%)")
print(f"Best Temp Scale Weight: {1.0 - best_w_iso:.2f} ({(1.0 - best_w_iso)*100:.0f}%)")
print(f"Best F1-Macro Score:    {best_f1:.4f}")

print("\n" + "="*60)
print("OPTIMAL BLENDED CLASSIFICATION REPORT")
print("="*60)
target_names = ['Clear Reply', 'Ambivalent', 'Clear Non-Reply']
print(classification_report(y_true, best_preds, target_names=target_names, digits=4))

In [ ]:
import numpy as np
import pandas as pd
from scipy.special import softmax
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, accuracy_score, classification_report

print("--- Starting Dirichlet Calibration ---")

# ==========================================
# 1. GET RAW PROBABILITIES
# ==========================================
# Assuming 'df' and 'y_true' are loaded
logits = df[['reply_logit', 'ambivalent_logit', 'nonreply_logit']].values
raw_probs = softmax(logits, axis=1)

# ==========================================
# 2. LOG-TRANSFORM THE PROBABILITIES
# ==========================================
# We add a tiny epsilon to prevent log(0) errors
eps = 1e-12
log_probs = np.log(raw_probs + eps)

# ==========================================
# 3. FIT THE DIRICHLET CALIBRATOR
# ==========================================
# We use a Multinomial Logistic Regression to replicate Dirichlet Calibration.
# C=10.0 applies very light regularization to prevent overfitting on our 308 samples.
dirichlet_calibrator = LogisticRegression(
    multi_class='multinomial',
    solver='lbfgs',
    C=10.0,
    max_iter=1000
)

# Fit the calibrator to learn the W matrix and b vector
dirichlet_calibrator.fit(log_probs, y_true)

# ==========================================
# 4. GENERATE CALIBRATED PROBABILITIES
# ==========================================
dir_calib_probs = dirichlet_calibrator.predict_proba(log_probs)

# Predict the class with the highest calibrated probability
dir_preds = np.argmax(dir_calib_probs, axis=1)

# ==========================================
# 5. EVALUATE METRICS
# ==========================================
f1_macro_dir = f1_score(y_true, dir_preds, average='macro')
acc_dir = accuracy_score(y_true, dir_preds)

print("\n" + "="*60)
print("DIRICHLET CALIBRATION METRICS")
print("="*60)
print(f"F1-MACRO SCORE: {f1_macro_dir:.4f}")
print(f"ACCURACY:       {acc_dir:.4f}")

print("\n" + "="*60)
print("DIRICHLET CLASSIFICATION REPORT")
print("="*60)
target_names = ['Clear Reply', 'Ambivalent', 'Clear Non-Reply']
print(classification_report(y_true, dir_preds, target_names=target_names, digits=4))

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, accuracy_score, classification_report

print("--- Starting Post-hoc Logit Adjustment ---")

# ==========================================
# 1. SETUP RAW LOGITS & PRIORS
# ==========================================
# Assuming 'df' and 'y_true' are still loaded
logits = df[['reply_logit', 'ambivalent_logit', 'nonreply_logit']].values

# Define the exact class counts from your dataset
class_counts = np.array([79, 206, 23])
priors = class_counts / class_counts.sum()

# Calculate the natural log of the priors
# Ambivalent will have the smallest penalty (closer to 0), Non-Reply the largest
log_priors = np.log(priors)

print(f"Class Priors: {priors}")
print(f"Log Priors (The Penalties): {log_priors}\n")

# ==========================================
# 2. GRID SEARCH FOR TAU
# ==========================================
best_f1 = 0.0
best_tau = 0.0
best_preds = None

# Search tau from 0.0 (no adjustment) to 2.0 (aggressive adjustment)
tau_range = np.arange(0.0, 2.0, 0.05)

for tau in tau_range:
    # Subtract the scaled log_priors from the logits
    adjusted_logits = logits - (tau * log_priors)

    # Predict based on the highest adjusted logit
    preds = np.argmax(adjusted_logits, axis=1)

    current_f1 = f1_score(y_true, preds, average='macro')

    if current_f1 > best_f1:
        best_f1 = current_f1
        best_tau = tau
        best_preds = preds

# ==========================================
# 3. EVALUATE METRICS
# ==========================================
print("="*60)
print("OPTIMAL LOGIT ADJUSTMENT FOUND")
print("="*60)
print(f"Best Tau:            {best_tau:.2f}")
print(f"Best F1-Macro Score: {best_f1:.4f}")

print("\n" + "="*60)
print("ADJUSTED CLASSIFICATION REPORT")
print("="*60)
target_names = ['Clear Reply', 'Ambivalent', 'Clear Non-Reply']
print(classification_report(y_true, best_preds, target_names=target_names, digits=4))

In [ ]:
import numpy as np
import pandas as pd
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import f1_score, accuracy_score, classification_report
from sklearn.model_selection import cross_val_predict

print("--- Starting KNN on Logit Manifold ---")

# ==========================================
# 1. SETUP
# ==========================================
# Assuming 'df' and 'y_true' are loaded
X_logits = df[['reply_logit', 'ambivalent_logit', 'nonreply_logit']].values
y = np.array(y_true)

# ==========================================
# 2. TEST DIFFERENT NEIGHBORHOODS
# ==========================================
# We will test a few different K values.
# We use cross_val_predict to prevent the KNN from just memorizing the exact point (which would give a fake 1.0 F1)
k_values = [3, 5, 7, 11, 15]
best_f1_knn = 0.0
best_k = 0
best_knn_preds = None

for k in k_values:
    # 'distance' weight means closer neighbors count more
    knn = KNeighborsClassifier(n_neighbors=k, weights='distance')

    # Cross-validation prediction ensures we test how well it generalizes locally
    knn_preds = cross_val_predict(knn, X_logits, y, cv=5)

    current_f1 = f1_score(y, knn_preds, average='macro')
    print(f"Testing K={k:2} | F1-Macro: {current_f1:.4f}")

    if current_f1 > best_f1_knn:
        best_f1_knn = current_f1
        best_k = k
        best_knn_preds = knn_preds

# ==========================================
# 3. RESULTS
# ==========================================
print("\n" + "="*60)
print(f"OPTIMAL KNN FOUND (K={best_k})")
print("="*60)
print(f"Best F1-Macro Score: {best_f1_knn:.4f}")

target_names = ['Clear Reply', 'Ambivalent', 'Clear Non-Reply']
print(classification_report(y, best_knn_preds, target_names=target_names, digits=4))

In [ ]:
import numpy as np
import pandas as pd
import torch
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import f1_score, accuracy_score, classification_report

print("--- Starting Embedding Extraction (Bypassing the Head) ---")

# ==========================================
# 1. EXTRACT 768D HIDDEN STATES
# ==========================================
# Assuming 'dataloader', 'model', and 'device' are loaded
all_embeddings = []

# Tell PyTorch to return the hidden states
model.eval()
with torch.no_grad():
    for batch in dataloader:
        inputs = {k: v.to(device) for k, v in batch.items()}
        # output_hidden_states=True is the magic key here
        outputs = model(**inputs, output_hidden_states=True)

        # We grab the very last hidden layer [-1]
        # And we grab the first token [:, 0, :] which is the [CLS] token representing the whole sequence
        cls_embeddings = outputs.hidden_states[-1][:, 0, :].cpu().numpy()
        all_embeddings.append(cls_embeddings)

X_embeddings = np.concatenate(all_embeddings, axis=0)
print(f"Successfully extracted embeddings. Shape: {X_embeddings.shape}")

# ==========================================
# 2. FIT NON-LINEAR CLASSIFIERS (SVM & Random Forest)
# ==========================================
# Assuming y_true is loaded from previous cells
y = np.array(y_true)

# Model 1: Support Vector Machine (RBF Kernel handles complex curves)
# class_weight='balanced' automatically handles your Ambivalent vs Clear Reply imbalance!
svm = SVC(kernel='rbf', class_weight='balanced', C=1.0)

# Model 2: Random Forest (Great for finding specific feature splits in high dimensions)
rf = RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42)

print("\n--- Running Cross-Validation on Embeddings ---")
svm_preds = cross_val_predict(svm, X_embeddings, y, cv=5)
rf_preds = cross_val_predict(rf, X_embeddings, y, cv=5)

# ==========================================
# 3. EVALUATE METRICS
# ==========================================
f1_svm = f1_score(y, svm_preds, average='macro')
f1_rf = f1_score(y, rf_preds, average='macro')

print("\n" + "="*60)
print("EMBEDDING-SPACE CLASSIFIER RESULTS")
print("="*60)
print(f"SVM (RBF Kernel) F1-Macro:     {f1_svm:.4f}")
print(f"Random Forest F1-Macro:        {f1_rf:.4f}")

# Display the report for the winner
best_preds = svm_preds if f1_svm > f1_rf else rf_preds
winner_name = "SVM" if f1_svm > f1_rf else "Random Forest"

print("\n" + "="*60)
print(f"WINNING MODEL CLASSIFICATION REPORT: {winner_name}")
print("="*60)
target_names = ['Clear Reply', 'Ambivalent', 'Clear Non-Reply']
print(classification_report(y, best_preds, target_names=target_names, digits=4))

In [ ]:
import gc
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import f1_score, accuracy_score, classification_report

print("--- Clearing GPU Memory ---")
# 1. FORCE CLEAR THE CRASHED MEMORY
torch.cuda.empty_cache()
gc.collect()

print("--- Starting Memory-Safe Embedding Extraction ---")

# 2. REDUCE BATCH SIZE TEMPORARILY
# Hidden states are massive. We drop batch size to 2 to prevent OOM.
SAFE_BATCH_SIZE = 2
# Assuming 'encoded_dataset' and 'data_collator' are still in memory from earlier
safe_dataloader = DataLoader(encoded_dataset, batch_size=SAFE_BATCH_SIZE, collate_fn=data_collator)

all_embeddings = []

# 3. EXTRACT WITH AGGRESSIVE MEMORY MANAGEMENT
model.eval()
with torch.no_grad():
    for step, batch in enumerate(safe_dataloader):
        inputs = {k: v.to(device) for k, v in batch.items()}

        outputs = model(**inputs, output_hidden_states=True)

        # Grab the [CLS] token from the final hidden layer and immediately move to CPU
        cls_embeddings = outputs.hidden_states[-1][:, 0, :].cpu().numpy()
        all_embeddings.append(cls_embeddings)

        # Manually delete large tensors and clear cache every 50 steps
        del outputs
        del inputs
        if step % 50 == 0:
            torch.cuda.empty_cache()

X_embeddings = np.concatenate(all_embeddings, axis=0)
print(f"Successfully extracted embeddings. Shape: {X_embeddings.shape}")

# ==========================================
# 4. FIT NON-LINEAR CLASSIFIERS
# ==========================================
# Assuming y_true is loaded
y = np.array(y_true)

print("\n--- Running Cross-Validation on Embeddings ---")
svm = SVC(kernel='rbf', class_weight='balanced', C=1.0)
rf = RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42)

svm_preds = cross_val_predict(svm, X_embeddings, y, cv=5)
rf_preds = cross_val_predict(rf, X_embeddings, y, cv=5)

# ==========================================
# 5. EVALUATE METRICS
# ==========================================
f1_svm = f1_score(y, svm_preds, average='macro')
f1_rf = f1_score(y, rf_preds, average='macro')

print("\n" + "="*60)
print("EMBEDDING-SPACE CLASSIFIER RESULTS")
print("="*60)
print(f"SVM (RBF Kernel) F1-Macro:     {f1_svm:.4f}")
print(f"Random Forest F1-Macro:        {f1_rf:.4f}")

best_preds = svm_preds if f1_svm > f1_rf else rf_preds
winner_name = "SVM" if f1_svm > f1_rf else "Random Forest"

print("\n" + "="*60)
print(f"WINNING MODEL CLASSIFICATION REPORT: {winner_name}")
print("="*60)
target_names = ['Clear Reply', 'Ambivalent', 'Clear Non-Reply']
print(classification_report(y, best_preds, target_names=target_names, digits=4))

In [ ]:
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import f1_score, accuracy_score, classification_report

print("--- Starting Monte Carlo Dropout Inference ---")

# ==========================================
# 1. ENABLE ONLY DROPOUT LAYERS
# ==========================================
# We keep the model in eval() to protect LayerNorm,
# but forcefully turn Dropout modules back to train() mode.
model.eval()
for m in model.modules():
    if m.__class__.__name__.startswith('Dropout'):
        m.train()

# ==========================================
# 2. RUN MULTIPLE FORWARD PASSES
# ==========================================
NUM_PASSES = 10
mc_logits_list = []

# Using your memory-safe dataloader from the last step
SAFE_BATCH_SIZE = 4
safe_dataloader = DataLoader(encoded_dataset, batch_size=SAFE_BATCH_SIZE, collate_fn=data_collator)

with torch.no_grad():
    for i in range(NUM_PASSES):
        print(f"Running MC Pass {i+1}/{NUM_PASSES}...")
        pass_logits = []

        for batch in safe_dataloader:
            inputs = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**inputs)
            pass_logits.append(outputs.logits.cpu().numpy())

        mc_logits_list.append(np.concatenate(pass_logits, axis=0))

# ==========================================
# 3. AVERAGE THE CHAOS
# ==========================================
# mc_logits_list is a list of 10 arrays. We average them across the passes.
mc_logits_stacked = np.stack(mc_logits_list)
final_mc_logits = np.mean(mc_logits_stacked, axis=0)

# Optional: You can also calculate the variance (uncertainty) to see which predictions the model panicked on!
# epistemic_uncertainty = np.var(mc_logits_stacked, axis=0)

# ==========================================
# 4. PREDICT & EVALUATE
# ==========================================
mc_preds = np.argmax(final_mc_logits, axis=1)

f1_mc = f1_score(y_true, mc_preds, average='macro')
acc_mc = accuracy_score(y_true, mc_preds)

print("\n" + "="*60)
print("MONTE CARLO DROPOUT METRICS")
print("="*60)
print(f"F1-MACRO SCORE: {f1_mc:.4f}")
print(f"ACCURACY:       {acc_mc:.4f}")

print("\n" + "="*60)
print("MC DROPOUT CLASSIFICATION REPORT")
print("="*60)
target_names = ['Clear Reply', 'Ambivalent', 'Clear Non-Reply']
print(classification_report(y_true, mc_preds, target_names=target_names, digits=4))

In [ ]:
import numpy as np
import pandas as pd
from scipy.special import softmax
from scipy.optimize import minimize
from sklearn.metrics import log_loss, f1_score, accuracy_score, classification_report

print("--- Starting Vector Scaling Calibration ---")

# ==========================================
# 1. LOAD RAW LOGITS & LABELS
# ==========================================
# Assuming 'df' and 'y_true' are still loaded from your baseline inference
logits = df[['reply_logit', 'ambivalent_logit', 'nonreply_logit']].values

# ==========================================
# 2. DEFINE THE VECTOR SCALING FUNCTION
# ==========================================
# We have 3 classes, so we need 3 weights and 3 biases = 6 parameters
def evaluate_vector_scaling(params, logits, y_true):
    W = params[0:3]
    b = params[3:6]

    # Apply W and b to each class independently
    scaled_logits = (logits * W) + b

    # Convert to probabilities
    scaled_probs = softmax(scaled_logits, axis=1)

    # Calculate Log-Loss
    # Add a tiny epsilon to prevent log(0)
    eps = 1e-15
    scaled_probs = np.clip(scaled_probs, eps, 1 - eps)
    return log_loss(y_true, scaled_probs)

# ==========================================
# 3. OPTIMIZE WEIGHTS AND BIASES
# ==========================================
# Initialize with W=1.0 and b=0.0 (which equals doing nothing)
initial_params = [1.0, 1.0, 1.0, 0.0, 0.0, 0.0]

# Run the optimizer
opt_result = minimize(
    evaluate_vector_scaling,
    initial_params,
    args=(logits, y_true),
    method='L-BFGS-B'
)

optimal_params = opt_result.x
opt_W = optimal_params[0:3]
opt_b = optimal_params[3:6]

print(f"✅ Optimal Weights (W): {opt_W}")
print(f"✅ Optimal Biases (b):  {opt_b}")

# ==========================================
# 4. APPLY AND PREDICT
# ==========================================
final_scaled_logits = (logits * opt_W) + opt_b
final_scaled_probs = softmax(final_scaled_logits, axis=1)

# Predict based on highest calibrated probability
vs_preds = np.argmax(final_scaled_probs, axis=1)

# ==========================================
# 5. METRICS
# ==========================================
f1_vs = f1_score(y_true, vs_preds, average='macro')
acc_vs = accuracy_score(y_true, vs_preds)

print("\n" + "="*60)
print("VECTOR SCALING METRICS")
print("="*60)
print(f"F1-MACRO SCORE: {f1_vs:.4f}")
print(f"ACCURACY:       {acc_vs:.4f}")

print("\n" + "="*60)
print("VECTOR SCALING CLASSIFICATION REPORT")
print("="*60)
target_names = ['Clear Reply', 'Ambivalent', 'Clear Non-Reply']
print(classification_report(y_true, vs_preds, target_names=target_names, digits=4))

In [ ]:
import numpy as np
import pandas as pd
from scipy.special import softmax
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import label_binarize
from sklearn.metrics import f1_score, accuracy_score, classification_report

print("--- Starting Beta Calibration ---")

# ==========================================
# 1. SETUP LOGITS & PROBS
# ==========================================
# Assuming 'df' and 'y_true' are loaded
logits = df[['reply_logit', 'ambivalent_logit', 'nonreply_logit']].values
raw_probs = softmax(logits, axis=1)

# Clip probabilities to prevent log(0)
eps = 1e-12
raw_probs = np.clip(raw_probs, eps, 1 - eps)

y_true_bin = label_binarize(y_true, classes=[0, 1, 2])
beta_calib_probs = np.zeros_like(raw_probs)

# ==========================================
# 2. FIT BETA CALIBRATION (ONE-VS-REST)
# ==========================================
for class_idx in range(3):
    p = raw_probs[:, class_idx]

    # Create the Beta Calibration features: ln(p) and ln(1-p)
    X_beta = np.column_stack([np.log(p), -np.log(1 - p)])

    # Fit a Logistic Regression to learn the a, b, c parameters
    lr = LogisticRegression(solver='lbfgs', C=999.0) # High C for unregularized fit
    lr.fit(X_beta, y_true_bin[:, class_idx])

    # Predict the calibrated probability for this class
    beta_calib_probs[:, class_idx] = lr.predict_proba(X_beta)[:, 1]

# ==========================================
# 3. NORMALIZE & EVALUATE
# ==========================================
row_sums = beta_calib_probs.sum(axis=1)[:, np.newaxis]
beta_calib_probs_norm = beta_calib_probs / row_sums

beta_preds = np.argmax(beta_calib_probs_norm, axis=1)

f1_beta = f1_score(y_true, beta_preds, average='macro')
acc_beta = accuracy_score(y_true, beta_preds)

print("\n" + "="*60)
print("BETA CALIBRATION METRICS")
print("="*60)
print(f"F1-MACRO SCORE: {f1_beta:.4f}")
print(f"ACCURACY:       {acc_beta:.4f}")

target_names = ['Clear Reply', 'Ambivalent', 'Clear Non-Reply']
print(classification_report(y_true, beta_preds, target_names=target_names, digits=4))

In [ ]:
import numpy as np
import pandas as pd
from scipy.special import softmax
from scipy.stats import entropy
from sklearn.metrics import f1_score, accuracy_score, classification_report

print("--- Starting Dynamic Temperature Scaling ---")

# ==========================================
# 1. CALCULATE INSTANCE-LEVEL ENTROPY
# ==========================================
logits = df[['reply_logit', 'ambivalent_logit', 'nonreply_logit']].values
raw_probs = softmax(logits, axis=1)

# Calculate entropy for each individual prediction
instance_entropy = entropy(raw_probs, axis=1, base=2)

# Normalize entropy to a 0.0 - 1.0 scale (Max entropy for 3 classes is ~1.585)
normalized_entropy = instance_entropy / np.log2(3)

# ==========================================
# 2. APPLY DYNAMIC TEMPERATURE
# ==========================================
# We map the entropy to a Temperature range.
# Confident predictions get T near 0.5 (sharper)
# Confused predictions get T near 2.5 (softer)
# You can tweak these min/max boundaries!
T_min = 0.5
T_max = 2.5

dynamic_T = T_min + (normalized_entropy * (T_max - T_min))
dynamic_T = dynamic_T[:, np.newaxis] # Reshape for broadcasting

# Apply the unique T to each row's logits
dynamic_scaled_logits = logits / dynamic_T
dynamic_probs = softmax(dynamic_scaled_logits, axis=1)

dynamic_preds = np.argmax(dynamic_probs, axis=1)

# ==========================================
# 3. EVALUATE
# ==========================================
f1_dyn = f1_score(y_true, dynamic_preds, average='macro')
acc_dyn = accuracy_score(y_true, dynamic_preds)

print("\n" + "="*60)
print("DYNAMIC TEMPERATURE SCALING METRICS")
print("="*60)
print(f"F1-MACRO SCORE: {f1_dyn:.4f}")
print(f"ACCURACY:       {acc_dyn:.4f}")

target_names = ['Clear Reply', 'Ambivalent', 'Clear Non-Reply']
print(classification_report(y_true, dynamic_preds, target_names=target_names, digits=4))

In [ ]:
import numpy as np
import pandas as pd
from scipy.special import softmax
from sklearn.metrics import f1_score, accuracy_score, classification_report

print("--- Starting Cascading Hierarchical Inference ---")

# ==========================================
# 1. SETUP RAW LOGITS
# ==========================================
logits = df[['reply_logit', 'ambivalent_logit', 'nonreply_logit']].values
raw_probs = softmax(logits, axis=1)

# ==========================================
# 2. STEP 1: ISOLATE "CLEAR NON-REPLY"
# ==========================================
# We set a threshold for Class 2. If it's above this, we lock it in.
# Default is 0.5, but we can tune it.
NON_REPLY_THRESH = 0.40
is_non_reply = raw_probs[:, 2] > NON_REPLY_THRESH

# ==========================================
# 3. STEP 2: BINARY SHOWDOWN (REPLY VS AMBIVALENT)
# ==========================================
# For the ones that are NOT Non-Replies, we ignore the Class 2 logit entirely.
# We recalculate a pure binary softmax using ONLY logits 0 and 1.
binary_logits = logits[:, :2]
binary_probs = softmax(binary_logits, axis=1)

# ==========================================
# 4. GRID SEARCH THE BINARY THRESHOLD
# ==========================================
best_f1_cascade = 0.0
best_binary_thresh = 0.0
best_cascade_preds = None

# We search for the exact tipping point between Reply (0) and Ambivalent (1)
for thresh in np.arange(0.1, 0.9, 0.01):
    cascade_preds = np.ones(len(logits), dtype=int) # Default to Ambivalent (1)

    # Apply Step 1 (Lock in Non-Replies)
    cascade_preds[is_non_reply] = 2

    # Apply Step 2 (Binary showdown for the rest)
    # If binary prob of 'Clear Reply' > thresh, set to 0
    mask_not_nonreply = ~is_non_reply
    is_clear_reply = (binary_probs[:, 0] > thresh) & mask_not_nonreply

    cascade_preds[is_clear_reply] = 0

    current_f1 = f1_score(y_true, cascade_preds, average='macro')

    if current_f1 > best_f1_cascade:
        best_f1_cascade = current_f1
        best_binary_thresh = thresh
        best_cascade_preds = cascade_preds.copy()

print("\n" + "="*60)
print("CASCADING HIERARCHICAL METRICS")
print("="*60)
print(f"Optimal Binary Threshold (Reply vs Ambiv): {best_binary_thresh:.2f}")
print(f"F1-MACRO SCORE: {best_f1_cascade:.4f}")

target_names = ['Clear Reply', 'Ambivalent', 'Clear Non-Reply']
print(classification_report(y_true, best_cascade_preds, target_names=target_names, digits=4))

In [ ]:
import numpy as np
from sklearn.metrics import f1_score, accuracy_score, classification_report

print("--- Starting Threshold Search on Calibrated Blend ---")

# ==========================================
# 1. RECREATE THE 0.7707 BLEND
# ==========================================
# Assuming iso_probs_normalized and ts_probs are still in memory from earlier
w_iso = 0.59
w_ts = 0.41

blended_probs = (w_iso * iso_probs_normalized) + (w_ts * ts_probs)

# ==========================================
# 2. GRID SEARCH CUSTOM MARGINS
# ==========================================
best_blend_f1 = 0.0
best_margin = 0.0
best_blend_preds = None

# Instead of argmax, we calculate the margin between Reply (0) and Ambivalent (1)
prob_reply = blended_probs[:, 0]
prob_ambiv = blended_probs[:, 1]
prob_nonrep = blended_probs[:, 2]

# We test different margin offsets
for offset in np.arange(-0.20, 0.20, 0.01):
    preds = np.ones(len(blended_probs), dtype=int) # Default Ambiv

    # Class 2 is highest
    is_nonreply = (prob_nonrep > prob_reply) & (prob_nonrep > prob_ambiv)
    preds[is_nonreply] = 2

    # For the rest, apply the offset to favor Clear Reply
    is_reply = (prob_reply + offset > prob_ambiv) & (~is_nonreply)
    preds[is_reply] = 0

    current_f1 = f1_score(y_true, preds, average='macro')

    if current_f1 > best_blend_f1:
        best_blend_f1 = current_f1
        best_margin = offset
        best_blend_preds = preds.copy()

print("\n" + "="*60)
print("THRESHOLDED BLEND METRICS")
print("="*60)
print(f"Optimal Probability Offset: {best_margin:+.2f}")
print(f"F1-MACRO SCORE: {best_blend_f1:.4f}")

target_names = ['Clear Reply', 'Ambivalent', 'Clear Non-Reply']
print(classification_report(y_true, best_blend_preds, target_names=target_names, digits=4))